# Addestramento del modello AI di Kurva3D

Menu *Runtime → Cambia tipo di runtime → GPU T4*, poi esegui le celle in ordine. Con la T4 gratuita servono circa 1-2 ore.

Base: Qwen3 4B (Apache 2.0). Strumenti: unsloth, PEFT, TRL (Apache 2.0), llama.cpp (MIT).

**Due tipi di addestramento** (scegli con `MODO` nella prima cella):
- `"parti"` (consigliato): oggetti complessi come violino, bicicletta, elicottero. Il modello impara a fare il piano e poi a costruire una parte per volta, come fa Kurva3D.
- `"semplice"`: il vecchio addestramento, un oggetto in un solo blocco di comandi.

In [ ]:
!nvidia-smi
MODO = "parti"   # "parti" oppure "semplice"
BRANCH = "main"
!rm -rf kurva3d && git clone --depth 1 -b $BRANCH https://github.com/mattialettere/rhino.git kurva3d

In [ ]:
!pip install -q unsloth

In [ ]:
if MODO == "parti":
    !python kurva3d/tools/ai-train/generate_parts_dataset.py --count 400 --out kurva3d-train.jsonl
else:
    !python kurva3d/tools/ai-train/generate_dataset.py --count 3000 --out kurva3d-train.jsonl
!head -c 1500 kurva3d-train.jsonl

In [ ]:
OUT = "kurva3d-qwen3-4b-parti" if MODO == "parti" else "kurva3d-qwen3-4b"
PARTS = "--parts" if MODO == "parti" else ""
!python kurva3d/tools/ai-train/train_lora.py $PARTS --data kurva3d-train.jsonl --base unsloth/Qwen3-4B --out $OUT

In [ ]:
import glob
from google.colab import files
for f in glob.glob(OUT + '/*.gguf'):
    print(f)
    files.download(f)